# Engine Size, Vehicle Class, and Fuel Consumption in Canada
## Phase 2 - Data Wrangling

**Author:** Matthew Meede  
**Scope:** Model years 2015-2024

### Project purpose
This notebook prepares the Natural Resources Canada fuel consumption
dataset for analysis of the relationship between engine size, vehicle
class, and combined fuel consumption among gasoline vehicles.

The preparation includes inspecting the original data, documenting
cleaning decisions, grouping SUV class labels consistently, creating
derived metrics, and validating the final analytical dataset.

## Data Acquisition

The original dataset is Natural Resources Canada's 2015–2024 Fuel
Consumption Ratings CSV.

Source: https://open.canada.ca/data/dataset/98f1a129-f628-4ce4-b24d-6f16bf24dd64

The data is loaded without modifying the original file. The first five
rows are displayed below to show its initial structure.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../Data/my2015-2024-fuel-consumption-ratings.csv")

raw = pd.read_csv(data_path)

raw.head()

,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191,NaN,NaN
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214,NaN,NaN
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140,NaN,NaN
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255,NaN,NaN
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244,NaN,NaN


## Initial Exploration

This section examines the dataset dimensions, column types, missing
values, numerical summaries, and vehicle class and fuel type labels.
These checks will guide the cleaning decisions.

In [2]:
print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")

raw.info()

Rows: 10,060
Columns: 15
<class 'pandas.DataFrame'>
RangeIndex: 10060 entries, 0 to 10059
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model year            10060 non-null  int64  
 1   Make                  10060 non-null  str    
 2   Model                 10060 non-null  str    
 3   Vehicle class         10060 non-null  str    
 4   Engine size (L)       10060 non-null  float64
 5   Cylinders             10060 non-null  int64  
 6   Transmission          10060 non-null  str    
 7   Fuel type             10060 non-null  str    
 8   City (L/100 km)       10060 non-null  float64
 9   Highway (L/100 km)    10060 non-null  float64
 10  Combined (L/100 km)   10060 non-null  float64
 11  Combined (mpg)        10060 non-null  int64  
 12  CO2 emissions (g/km)  10060 non-null  int64  
 13  CO2 rating            8932 non-null   float64
 14  Smog rating           7826 non-null   float64
dtypes: fl

In [3]:
display(
    raw.isna().sum()
    .rename("Missing values")
    .to_frame()
)

display(raw.describe())

display(raw["Vehicle class"].value_counts(dropna=False))
display(raw["Fuel type"].value_counts(dropna=False))

print("Exact duplicate rows:", raw.duplicated().sum())

,Missing values
Model year,0
Make,0
Model,0
Vehicle class,0
Engine size (L),0
Cylinders,0
Transmission,0
Fuel type,0
City (L/100 km),0
Highway (L/100 km),0


,Model year,Engine size (L),Cylinders,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
count,10060.000000,10060.000000,10060.00000,10060.000000,10060.000000,10060.000000,10060.000000,10060.000000,8932.000000,7826.000000
mean,2019.242048,3.144901,5.61839,12.483052,9.142803,10.979712,27.436282,253.711332,4.617443,4.816509
std,2.834109,1.340010,1.87591,3.404483,2.178649,2.808468,7.316270,60.474917,1.574763,1.783793
min,2015.000000,0.900000,3.00000,4.000000,3.900000,4.000000,11.000000,94.000000,1.000000,1.000000
25%,2017.000000,2.000000,4.00000,10.100000,7.600000,9.000000,22.000000,210.000000,4.000000,3.000000
50%,2019.000000,3.000000,6.00000,12.100000,8.900000,10.700000,26.000000,250.000000,5.000000,5.000000
75%,2022.000000,3.700000,6.00000,14.500000,10.400000,12.700000,31.000000,292.000000,5.000000,6.000000
max,2024.000000,8.400000,16.00000,30.700000,20.900000,26.100000,71.000000,608.000000,10.000000,8.000000


Vehicle class
Sport utility vehicle: Small       1868
Mid-size                           1393
Sport utility vehicle: Standard    1231
Compact                            1135
Pickup truck: Standard              878
Subcompact                          850
Full-size                           779
Two-seater                          639
Minicompact                         460
Station wagon: Small                282
Pickup truck: Small                 220
Special purpose vehicle             103
Minivan                              95
Station wagon: Mid-size              82
Van: Passenger                       45
Name: count, dtype: int64

Fuel type
X    4756
Z    4739
E     328
D     236
N       1
Name: count, dtype: int64

Exact duplicate rows: 0


## Data Cleaning

The original dataset contains 10,060 rows and 15 columns, with no exact
duplicate rows. Model year, engine size, vehicle class, fuel type, and
combined fuel consumption have no missing values.

CO2 rating has 1,128 missing values and Smog rating has 2,234.
These optional ratings are not needed for this investigation, so they
will be excluded from the analytical dataset rather than filled with
invented values or used to remove otherwise complete records.

The analytical dataset will retain regular gasoline (X) and premium
gasoline (Z) records for model years 2015–2024. Gasoline hybrid records
remain included, consistent with the proposal. Other fuel types are
excluded to match the study scope.

The original dataset will remain unchanged.

In [5]:
# Work on a copy to preserve the original data.
clean = raw.copy()

# Remove surrounding whitespace from text columns and column headings.
clean.columns = clean.columns.str.strip()

text_columns = [
    "Make", "Model", "Vehicle class", "Transmission", "Fuel type"
]

for column in text_columns:
    clean[column] = clean[column].str.strip()

rows_before = len(clean)

# Select the study's model years and gasoline fuel types.
in_scope = (
    clean["Model year"].between(2015, 2024)
    & clean["Fuel type"].isin(["X", "Z"])
)

clean = clean.loc[in_scope].copy()

# Exclude optional ratings that are outside the research question.
clean = clean.drop(columns=["CO2 rating", "Smog rating"])

print(f"Original rows: {rows_before:,}")
print(f"Gasoline rows retained: {len(clean):,}")
print(f"Rows excluded by scope: {rows_before - len(clean):,}")

display(clean.head())

Original rows: 10,060
Gasoline rows retained: 9,495
Rows excluded by scope: 565


,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km)
0,2015,Acura,ILX,Compact,2.0,4,AS5,Z,9.7,6.7,8.3,34,191
1,2015,Acura,ILX,Compact,2.4,4,M6,Z,10.8,7.4,9.3,30,214
2,2015,Acura,ILX Hybrid,Compact,1.5,4,AV7,Z,6.0,6.1,6.1,46,140
3,2015,Acura,MDX SH-AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,2015,Acura,RDX AWD,Sport utility vehicle: Small,3.5,6,AS6,Z,12.1,8.7,10.6,27,244
